$$\large{\color{yellow}{\text{MDS6304 Deep Learning Webinar-7}}}$$

$$\large\text{Theme}: \underline{\text{advanced tensor operations for calculating backward propagating gradients in a deep learning model}}$$

---

**Load essential libraries**

---

In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import sys
from sklearn.preprocessing import StandardScaler

--- 

Recall that a matrix-vector product is a sequence of dot products:

![Matrix-vector product](https://1drv.ms/i/c/37720f927b6ddc34/IQRqA-Ychrs2SLg73OG4mi0XAa7Q5wdpS0v1n-fj89og1kk?width=660)

This can be extended to a rank-3 tensor-vector product which can be seen as a sequence of matrix-vector products.

![Tensor-vector product](https://1drv.ms/i/c/37720f927b6ddc34/IQRew4AFe9nFSKFrk1bgsGybAbWQ-l51T8T4WoHxr1-zLrE?width=660)

---

In [5]:
A = torch.tensor([[1, 2, 4], [2, -1, 3]])
print(f'Matrix A = \n{A}')
print('-----')
v = torch.tensor([4, 2, -2])
print(f'Vector v = \n{v}')
print('-----')
print(f'Matrix A times vector v = \n{A @ v}')
print('-----')
print(f'Dot product of element-0 of matrix A (row-0) and vector v = \n{A[0] @ v}')
print('-----')
T = torch.tensor([
                  [[1, 2, 4], [2, -1, 3]],
                  [[3, 1, 0], [1, -4, 2]],
                  [[1, 5, -4], [3, -1, 2]] 
                ])
print(f'Tensor T = \n{T}')
print('-----')
print(f'Vector v = \n{v}')
print('-----')
print(f'Tensor T times matrix v = \n{T @ v}')
print('-----')
print(f'Dot product of element-0 of T and vector v = \n{T[0] @ v}')

Matrix A = 
tensor([[ 1,  2,  4],
        [ 2, -1,  3]])
-----
Vector v = 
tensor([ 4,  2, -2])
-----
Matrix A times vector v = 
tensor([0, 0])
-----
Dot product of element-0 of matrix A (row-0) and vector v = 
0
-----
Tensor T = 
tensor([[[ 1,  2,  4],
         [ 2, -1,  3]],

        [[ 3,  1,  0],
         [ 1, -4,  2]],

        [[ 1,  5, -4],
         [ 3, -1,  2]]])
-----
Vector v = 
tensor([ 4,  2, -2])
-----
Tensor T times matrix v = 
tensor([[ 0,  0],
        [14, -8],
        [22,  6]])
-----
Dot product of element-0 of T and vector v = 
tensor([0, 0])


---

Consider a dataset with 4 samples and 3 output categories. The true one-hot encoded output labels are:

$$\mathbf{Y} = \begin{bmatrix}{\color{red}{\mathbf{y}^{(0)}}^\mathrm{T}}\\{\color{green}{\mathbf{y}^{(1)}}^\mathrm{T}}\\{\color{blue}{\mathbf{y}^{(2)}}^\mathrm{T}}\\{\color{cyan}{\mathbf{y}^{(3)}}^\mathrm{T}}\end{bmatrix} = \begin{bmatrix}\boxed{\color{red}1.0}&\color{red}0.0&\color{red}0.0\\\color{green}0.0&\boxed{\color{green}1.0}&\color{green}0.0\\\color{blue}0.0&\boxed{\color{blue}1.0}&\color{blue}0.0\\\color{cyan}0.0&\color{cyan}0.0&\boxed{\color{cyan}1.0}\end{bmatrix}.$$

The softmax model results in the following predicted probabilities:

$$\mathbf{A} = \begin{bmatrix}{\color{red}{\mathbf{a}^{(0)}}^\mathrm{T}}\\{\color{green}{\mathbf{a}^{(1)}}^\mathrm{T}}\\{\color{blue}{\mathbf{a}^{(2)}}^\mathrm{T}}\\{\color{cyan}{\mathbf{a}^{(3)}}^\mathrm{T}}\end{bmatrix} = \begin{bmatrix}\boxed{\color{red}0.7}&\color{red}0.1&\color{red}0.2\\\color{green}0.1&\boxed{\color{green}0.3}&\color{green}0.6\\\color{blue}0.5&\boxed{\color{blue}0.4}&\color{blue}0.1\\\color{cyan}0.3&\color{cyan}0.3&\boxed{\color{cyan}0.4}\end{bmatrix}.$$

---

In [ ]:
# True one-hot encoded output labels matrix for a batch of 4 samples
# corresponding to 3 possible output categories
b = 4 # no. of samples in the batch
Y = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype = torch.float64)
print(f'True one-hot encoded output labels matrix for a batch of {b} samples = \n{Y}')
print('------')

# Predicted probabilities for a batch of 4 samples corresponding
# to 3 possible output categories
A = torch.tensor([[0.7, 0.1, 0.2], [0.1, 0.3, 0.6], [0.5, 0.4, 0.1], [0.3, 0.3, 0.4]], dtype = torch.float64)
print(f'Predicted probabilities for a batch of {b} samples = \n{A}')

---

Adding a new axis to the matrix of predicted probabilities $\mathbf{A}$.

---

In [ ]:
print(f'Predicted probabilities shape = {A.shape}')
print(f'Predicted probabilities = \n{A}')
print('-------')
print(f'Predicted probabilities shape with new axis added at the end = {A.unsqueeze(-1).shape}')
print(f'Predicted probabilities with new axis added at the end = \n{A.unsqueeze(-1)}')
print('-------')
print(f'Predicted probabilities shape with new axis added at the middle = {A.unsqueeze(-2).shape}')
print(f'Predicted probabilities with new axis added at the end = \n{A.unsqueeze(-2)}')

---

Demonstrating how softmax operates on batched data and how its gradients are structured in deep learning.

For each sample, calculate the local gradient of the softmax layer which is "how sensitive are the output softmax-activated scores w.r.t. the input raw scores" and assemble the gradients for all samples as a rank-3 tensor:

![Softmax gradient tensor](https://1drv.ms/i/c/37720f927b6ddc34/IQQtbZDgPL06S4vZZJmB7l5KAfv7iYz7MHHd1cGO8Bx0Kcc?width=660)

---

In [ ]:
# A 3x3 identity matrix corresponding to the number of output categories which is 
I = torch.eye(3)
print(f'Identity matrix I = \n{I}')
print('-------')

# Softmax local gradient for sample-0
print(f'Sample-0 predicted probabilities as a one column matrix = \n{A[0].reshape(-1, 1)}')
print(f'Sample-0 predicted probabilities as a one row matrix = \n{A[0].reshape(1, -1)}')
softmax_local_gradient_sample0 = (I - A[0].reshape(-1, 1)) * (A[0].reshape(1, -1))
print(f'Softmax local gradient for sample-0 = \n{softmax_local_gradient_sample0}')
print('-------')

# Softmax local gradient for all samples
softmax_local_gradient = (I - A.unsqueeze(-1)) * (A.unsqueeze(-2))
print(f'Softmax local gradient for all samples = \n{softmax_local_gradient}')

---

Understanding the einsum() function for effecient tensor-based calculations.

---

In [ ]:
# Softmax local gradient for all samples
print(f'Softmax local gradient for all samples = \n{softmax_local_gradient}')
print('-----')

# Loss gradient which is also the output side gradient of the softmax layer
loss_gradient = -Y/A
softmax_output_gradient = loss_gradient
print(f'Output side gradient of softmax layer for all samples = \n{loss_gradient}')
print('-----')

# Gradient flowing backward on the input side of the softmax layer
softmax_input_gradient = torch.einsum('ij, ijk -> ik', softmax_output_gradient, softmax_local_gradient)
print(f'Input side gradient of softmax layer for all samples = \n{softmax_input_gradient}')

---

How exactly does the einsum() function work in the following case?

$$\texttt{orch.einsum('ij, ijk -> ik', softmax\_output\_gradient, softmax\_local\_gradient)}$$

---

In [ ]:
# Gradient flowing backward on the input side of the softmax layer
# for sample-0
softmax_output_gradient[0].reshape(1, -1) @ softmax_local_gradient[0]

---

Using einsum() to calculate the gradient w.r.t. the weights of the dense layer.

---

In [ ]:
# Patients data matrix
X = torch.tensor([[72, 120, 37.3, 104, 32.5],
                 [85, 130, 37.0, 110, 14],
                 [68, 110, 38.5, 125, 34],
                 [90, 140, 38.0, 130, 26]], dtype = torch.float64)
sc = StandardScaler() # create a standard scaler object
X_std = torch.tensor(sc.fit_transform(X), dtype = torch.float64)
print(f'Standardized data matrix:\n {X_std}') 

# Initial Weights matrix (trainable tensor)
W = torch.tensor([[0.9, 0.5, 0.3],
                  [0.9, 0.3, 0.5],
                  [-1.5, 0.4, 0.1],
                  [0.1, 0.1, -1.0],
                  [-1.2, 0.5, -0.8]], dtype = torch.float64,
                 requires_grad = True)
print(f'Initial weights matrix:\n {W}')
print('------')

# Gradient w.r.t. the weights of the dense layer
dense_output_gradient = softmax_input_gradient
dense_local_gradient = X_std
print(f'Local gradient of the denser layer for all samples = \n{dense_local_gradient}')
print(f'Output side gradient of the denser layer for all samples = \n{dense_output_gradient}')
dense_weights_gradient = torch.einsum('ij, ik -> jk', dense_local_gradient, dense_output_gradient)
print(f'Gradient w.r.t. the weights of the dense layer for all samples = \n{dense_weights_gradient}')

---

How exactly does the einsum() function work in the following case?

$$\texttt{torch.einsum('ij, ik -> jk', dense\_local\_gradient, dense\_output\_gradient)}$$

---

In [ ]:
dense_local_gradient[0].reshape(-1, 1) @ dense_output_gradient[0].reshape(1, -1) +\
dense_local_gradient[1].reshape(-1, 1) @ dense_output_gradient[1].reshape(1, -1) +\
dense_local_gradient[2].reshape(-1, 1) @ dense_output_gradient[2].reshape(1, -1) +\
dense_local_gradient[3].reshape(-1, 1) @ dense_output_gradient[3].reshape(1, -1)

---

Final update of the weights of the dense layer.

---

In [ ]:
print(f'Initial weights matrix W:\n {W}')
print(f'Shape of the weights matrix W:\n {W.shape}')
print('---------')
print(f'Gradient w.r.t. the weights:\n {dense_weights_gradient}')
print(f'Shape of the gradient w.r.t. the weights:\n {dense_weights_gradient.shape}')
print('---------')

# Weights update using the gradient
lr = 1e-02 # learning 
with torch.no_grad():
    W += lr*(-dense_weights_gradient)

print(f'Updated weights matrix W:\n {W}')